# UNIQUE input data preparation: CLint

---

The process of preparing CLint input datasets to run UNIQUE Pipeline includes the following steps:
- Importing the preprocessed CLint dataset.
- Integrating predictions from Chemprop and corresponding uncertainty estimates (ensemble variance).
- Adding and scaling GNN latent fingerprints.

In [ ]:
import pandas as pd
pd.set_option('display.max_columns', None)
import numpy as np

## Load CLint data

In [ ]:
df = pd.read_csv('../data/CLint_dataset_with_splits.csv', index_col=0)
df.head()

In [ ]:
df['which_set'] = df['Subset'].copy()
df['which_set'] = df['which_set'].replace({'Training':'TRAIN', 'Calibration':'CALIBRATION', 'Validation':'TEST'})
df['which_set'].value_counts()

Select columns

In [ ]:
df = df[['Id','Structure','rLM LogCLint','hLM LogCLint','which_set']]
df.head()

## Read model predictions

In [ ]:
df_preds = pd.read_csv('../models/CLint_model/predictions.csv', index_col=0)
df_preds.head()

In [ ]:
all([df.Id.iloc[x] == df_preds.Id.iloc[x] for x in range(df.shape[0])])

Rename columns

In [ ]:
df_preds = df_preds.rename(columns={"pred_0": "predicted rLM LogCLint", "pred_0_unc": "variance rLM LogCLint",
                                    "pred_1": "predicted hLM LogCLint", "pred_1_unc": "variance hLM LogCLint"})

Select columns

In [ ]:
df_preds = df_preds[['Id','predicted rLM LogCLint', 'predicted hLM LogCLint','variance rLM LogCLint','variance hLM LogCLint']]
df_preds.head()

Merge predictions data

In [ ]:
print(df.shape)
print(df_preds.shape)
df = df.merge(df_preds, on=['Id'])
print(df.shape)
df.head()

## Read GNN latent representations

In [ ]:
df_latent = pd.read_csv('../models/CLint_model/latent_fps.csv')
df_latent.head()

Add SMILES data

In [ ]:
original_data = pd.read_csv('../data/CLint_dataset_with_splits.csv', index_col=0)
smiles = original_data['Structure']
smiles

In [ ]:
print(df_latent.shape)
print(smiles.shape)
df_latent = pd.concat([smiles, df_latent], axis=1)
print(df_latent.shape)
df_latent.head()

In [ ]:
# Drop duplicated latent profiles
df_latent = df_latent.drop_duplicates()
df_latent.shape

Inspect feature distributions and outlier identification

In [ ]:
import matplotlib.pyplot as plt
import scipy.stats as stats
import seaborn as sns

# inspect the first latent feature
feature = df_latent['fp_0']
# histogram
sns.histplot(feature, kde=True, bins=30)
plt.show()
# Q-Q plot
stats.probplot(feature, dist="norm", plot=plt)
plt.show()

In [ ]:
# inspect the second latent feature
feature = df_latent['fp_1']
# histogram
sns.histplot(feature, kde=True, bins=30)
plt.show()
# Q-Q plot
stats.probplot(feature, dist="norm", plot=plt)
plt.show()

In [ ]:
print(df_latent['fp_0'].skew(axis=0, skipna=True))
print(df_latent['fp_1'].skew())

In [ ]:
def detect_outliers(df, feature):
    zscores = (df[feature] - df[feature].mean())/df[feature].std(ddof=0)

    lower_outliers = df[zscores < -3]
    upper_outliers = df[zscores > 3]

    lower_percent = len(lower_outliers) / len(df) * 100
    upper_percent = len(upper_outliers) / len(df) * 100

    return lower_percent, upper_percent

In [ ]:
lower_percent, upper_percent = detect_outliers(df_latent, 'fp_0')
print(lower_percent, upper_percent)
lower_percent, upper_percent = detect_outliers(df_latent, 'fp_1')
print(lower_percent, upper_percent)

Add latent fingerprint descriptors in a single column and scale them

In [ ]:
df_latent['latent_fp'] = [np.array(df_latent.iloc[x, np.where(['fp_' in col for col in df_latent.columns])[0]]) for x in range(df_latent.shape[0])]
df_latent.head()

Merge latent representations

In [ ]:
set(df.Structure) == set(df_latent.Structure)

In [ ]:
print(df.shape)
print(df_latent.shape)
df = df.merge(df_latent[['Structure','latent_fp']], on=['Structure'])
print(df.shape)
df.head()

Given that most of latent fingerprint features show a positively skewed distribution (which does not follow a Gaussian distribution) and have a significant percentage of outliers (more than 0.3% as determined from the Z-score detection method), we employed a Robust Scaler to perform feature transformation, rather than using a Standard Scaler.

In [ ]:
from sklearn.preprocessing import RobustScaler
scaler = RobustScaler()
scaler.fit(np.array([np.array(x) for x in df[df.which_set == 'TRAIN'].latent_fp]))
df['latent_fp_scaled'] = [list(i) for i in scaler.transform(np.array([np.array(x) for x in df.latent_fp]))]
df = df.drop(columns='latent_fp')
df.head()

## Save final datasets

Save the generated UNIQUE input dataset for each endpoint into a separate CSV file.

In [ ]:
for endpoint in ['rLM LogCLint', 'hLM LogCLint']:

    print(f'\n{endpoint}')
    df_endpoint = df[['Id', 'Structure', f'{endpoint}', f'predicted {endpoint}', f'variance {endpoint}', 'which_set', 'latent_fp_scaled']]
    df_endpoint.columns = ['Id', 'Structure', 'labels', 'predictions', 'variance', 'which_set', 'latent_fp_scaled']

    print(df_endpoint.shape)
    df_endpoint = df_endpoint.dropna()
    print(df_endpoint.shape)

    print(df_endpoint.which_set.value_counts())
    print(df_endpoint.which_set.value_counts()/df_endpoint.which_set.value_counts().sum() * 100)

    df_endpoint.to_csv(f'../unique_input_data/{endpoint}_unique_input_data.csv')